# B2-024 — Session 2: Experiment Design Under a Budget

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261022`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C3-gradient-descent`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C7-cnn-transfer`, `book1:C10-competition-craft`, `book1:C11-neural-training`, `book1:C12-classical-models`, `B2-023-generative-models-diffusion`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C3-gradient-descent](../../../../book1/units/C3-gradient-descent/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C7-cnn-transfer](../../../../book1/units/C7-cnn-transfer/lesson.ipynb), [book1:C10-competition-craft](../../../../book1/units/C10-competition-craft/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [book1:C12-classical-models](../../../../book1/units/C12-classical-models/lesson.ipynb), [B2-023-generative-models-diffusion](../../B2-023-generative-models-diffusion/lesson.ipynb).

**Data boundary:** use literal unit data and the seeded unit generator (`scripts/generate_capstone_data.py`, read through `data/capstone_data.py`), with the standard library, NumPy, and Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints you did not create in the same run, or student data.

An open-ended Round 2 task gives you data, a metric, a deadline, and freedom.
Freedom is the hard part: with limited compute you can run perhaps a dozen honest experiments, so each one must answer a question.
This session turns "try things until the number improves" into a design: a falsifiable hypothesis, a baseline run under the same protocol, one-factor ablations, a fixed step budget, repeated seeds, held-out discipline, and a written log that decides what to run next.
Book 1 taught the locked-test protocol (`book1:hidden-test-protocol`) and score-driven iteration (`book1:metric-driven-iteration`); here they become a plan you can execute under a budget.

## 1. From a question to a falsifiable hypothesis

A useful experiment starts from a **hypothesis** with four parts:

1. **The change**: exactly one thing that differs between two runs ("feed log magnitudes instead of raw magnitudes").
2. **The metric and its direction**: which number moves, and which way ("validation mean source-position error decreases").
3. **The size that matters**: the smallest effect worth acting on ("by at least 10% relative"), chosen *before* running.
4. **The protocol**: data split, budget, and seeds that both arms share.

Written out: *"With the same 8→64→64→3 MLP, 300 Adam steps, and seeds 0–2, feeding log magnitudes instead of raw standardized magnitudes reduces the mean validation position error by at least 10%."*

A hypothesis is **falsifiable** when some outcome of the planned runs would refute it.
"The model will be better" is not; there is always some metric on which it is better.
Fixing the metric and the minimum effect in advance also protects you from the most common self-deception in open-ended work: running many variants and then reporting whichever metric happened to improve.

**Why the physics suggests this hypothesis.** In the inverse problem of Session 5 the magnitudes span roughly two orders of magnitude and the noise is multiplicative (each reading is the true value times $1+0.03\varepsilon$).
Taking logs turns multiplication into addition: $\log m = \log q - \log(r^2+h^2) + \log(1+0.03\varepsilon)$, so the noise becomes approximately additive with a constant scale, and the strength $q$ becomes an additive offset.
Good hypotheses usually come from a mechanism like this, not from a list of tricks.

**Checkpoint 1A.** Rewrite "bigger models are better" as a falsifiable hypothesis for the inverse task.

**Checkpoint 1B.** Why must the minimum effect size be chosen before the runs?

**Collected answers.** For example: "With log features, 300 Adam steps, and seeds 0–2, an MLP of hidden width 64 has a mean validation position error at least 10% lower than width 32." Because choosing it afterwards lets the observed difference set the bar, so any difference can be declared meaningful; fixing it first makes the test honest.

## 2. Baselines: the comparison that gives a number meaning

A score means nothing alone: "0.021 position error" is good or bad only relative to something.
There are three kinds of baselines, and a serious study reports at least two.

* **Trivial baseline**: a rule with no learning, such as predicting the center of the source domain $(0.5, 0.5)$ for every input, or the majority class. It tells you whether the task is being learned at all.
* **Committed baseline**: a fixed method someone else can rerun, such as this unit's grid-Tikhonov inversion, the fixed-initialization least-squares fit for mixtures, or the labelled-only CNN. `capstone_data.baseline_score(task)` computes these with committed code at the frozen seed, so every learner compares against the same number.
* **Strong internal baseline**: your own best simple model (for example the MLP before you add a trick). Ablations (Section 3) compare against it.

**A baseline must run under the same protocol** as the method it is compared with: the same training rows, the same validation rows, the same metric code, and a comparable budget.
A baseline trained on fewer rows, scored with a different metric implementation, or tuned less carefully makes any method look good.

**Weak baselines are honest information.** This unit's grid-Tikhonov baseline is weak (Session 5 explains why: 81 unknowns from 8 sensors), and the statements print its number rather than hiding it.
Beating a weak baseline proves only that you are not worse than it; the interesting comparisons are then between your own variants, under the protocol.

**Checkpoint 2A.** What is the trivial-baseline position error if the true sources were spread uniformly over the whole of $[0.1,0.9]^2$ and you always predict $(0.5,0.5)$? Give a bound rather than an exact value.

**Checkpoint 2B.** Your method uses 2,000 Adam steps; the internal baseline you compare with used 200. What is wrong with the comparison?

**Collected answers.** Every error is at most the distance from the center to a corner, $\sqrt{0.4^2+0.4^2}\approx0.566$, and it is $0$ only at the center, so the mean lies strictly between $0$ and $0.566$ (its exact value is about $0.31$, from integrating over the square). The budgets differ, so any improvement could come from ten times more training rather than from the method; both arms need the same step budget.

## 3. Ablations: change one thing

An **ablation** removes (or adds) exactly one component while holding everything else fixed, to measure that component's contribution.
If configuration $B$ differs from $A$ in two ways, say width *and* augmentation, a difference between them cannot be attributed to either.
This is **confounding**, and it is the most common design error in open-ended reports.

**Designing an ablation set.** Start from the full method $M$ and list its components $c_1,\dots,c_k$.
Run $M$, and $M$ without $c_j$ for each $j$ (leave-one-out), all with the same seeds and budget.
A component whose removal does not hurt beyond seed noise (Section 5) is not earning its complexity.
Alternatively, start from the baseline and add one component at a time (add-one-in); the two orders can disagree when components interact, which is itself worth reporting.

**Ablation versus tuning.** An ablation answers "does component $c$ matter?".
Tuning answers "what value of $c$'s knob is best?".
Mixing them inflates results: if you tune the full model carefully but run its ablations at default settings, the ablations look worse than they are.
Give every arm the same tuning effort, or none.

**Reading an ablation table.** Each row is a configuration, each column a statistic (mean, sample standard deviation over seeds, number of seeds, budget).
Practice p12 asks you to build such a table programmatically; p24 asks you to decide which differences in one are real.

**Checkpoint 3A.** A run changes the hidden width from 32 to 64 and turns on input-noise augmentation; validation error falls 15%. What can you conclude about augmentation?

**Checkpoint 3B.** Give the leave-one-out ablation set for a method with components log features, augmentation, and width 64 (versus 32).

**Collected answers.** Nothing specific: the two changes are confounded; the 15% could come from width, from augmentation, or from their interaction. Full method (log + aug + width 64); without log (raw + aug + 64); without augmentation (log + 64); without the width increase (log + aug + 32); all with the same seeds and budget.

## 4. Fixed compute budgets

Compute is the currency of an experiment, so it must be **fixed and counted**.
Count **optimizer steps** (or wall-clock time), not "epochs": an epoch's cost depends on the batch size and dataset size, so two configurations with "10 epochs" can differ in cost by an order of magnitude.

**The `StepBudget` wrapper.** The unit's data module provides `capstone_data.StepBudget(max_steps)`.
Wrap every optimizer you create with `budget.wrap(optimizer)`; each `step()` increments `budget.used`, and the step beyond `max_steps` raises `StepBudgetExceeded`.
`budget.elapsed()` reports wall-clock seconds since the budget was created.
Closed-form computations (a linear solve, k-means assignments, a forward pass) cost no steps; they still cost wall-clock time, which the challenge practices also cap.

**Fair comparisons.** When comparing configurations, give each the **same budget per run**: same number of steps, and for full-batch training the same data per step.
A configuration that "needs more steps" is a finding, recorded as a separate, labelled run, not a silent advantage.

**Planning a campaign.** With a total budget $S$ and $r$ seeds per configuration at $s$ steps per run, you can afford $\lfloor S/(rs)\rfloor$ configurations.
For $S=2000$, $r=2$, $s=300$: three configurations ($1{,}800$ steps), with $200$ steps of slack.
Write this arithmetic into the log *before* the first run; Practice p22 grades it.

**Checkpoint 4A.** With a 2,000-step budget, 3 seeds per configuration, and 250 steps per run, how many configurations can you compare?

**Checkpoint 4B.** Does a k-means clustering run inside a `StepBudget` consume budget steps?

**Collected answers.** $\lfloor 2000/(3\cdot250)\rfloor = 2$ configurations (1,500 steps, 500 left). No: it uses no optimizer `step()` calls; it still costs wall-clock time.

## 5. Seeds and repeated runs

A trained model's score depends on random choices: initialization, mini-batch order, augmentation noise.
`book1:random-seeding` taught you to fix seeds for reproducibility; here we use **several** seeds to measure how much the score moves for reasons unrelated to the change being tested.

**Per-configuration statistics.** For scores $s_1,\dots,s_r$ from $r$ seeds, report the mean $\bar s$ and the **sample standard deviation**
$$\hat\sigma=\sqrt{\frac{1}{r-1}\sum_{i=1}^r (s_i-\bar s)^2},$$
with the $r-1$ divisor (`torch.std` default, `unbiased=True`) because $\bar s$ was estimated from the same scores.
The population formula with divisor $r$ underestimates the spread, badly so for $r=2$ or $3$.

**Shared seeds pair the runs.** Use the *same* seed list for every configuration, so seed 0 of configuration $A$ and seed 0 of $B$ start from the same initialization (when architectures match) and see the same batch order.
The per-seed differences $d_i=s^B_i-s^A_i$ then cancel much of the shared randomness; Session 3 shows why the variance of a difference shrinks when the two runs are positively correlated, and Practice p15 proves it.

**How many seeds?** Enough that the effect you care about is larger than the seed spread.
A rough rule: if two configurations' means differ by less than about one sample standard deviation of the per-seed differences, you have not shown a difference, whatever the means say.
With a tight budget, two or three seeds per configuration and a pre-registered decision rule are better than one seed and a story.

**Seed cherry-picking.** Running five seeds and reporting the best is not a result; it is the maximum of five noisy draws, which is biased upward.
Report all seeds, or the mean and spread.

**Checkpoint 5A.** Scores over three seeds are $0.020$, $0.022$, $0.027$. Compute the mean and the sample standard deviation.

**Checkpoint 5B.** Why does reporting the best of five seeds overstate a method?

**Collected answers.** Mean $=0.023$; deviations $-0.003,-0.001,0.004$; squares sum $9+1+16=26$ (in units of $10^{-6}$); divide by $r-1=2$: $13\times10^{-6}$; $\hat\sigma=\sqrt{13}\times10^{-3}\approx0.0036$. The maximum of several noisy draws is systematically better than a typical draw, so the reported number is biased toward luck rather than the method's expected performance.

## 6. Held-out discipline and the experiment log

**Three splits, three roles** (`book1:train-test-split`, `book1:hidden-test-protocol`):

* **Training rows** fit parameters.
* **Validation rows** choose between configurations, thresholds, step counts, and every other decision.
* **The locked test set** is scored **once**, after all fitting and selection, to report the final number.

Each time a decision uses a set, information from it leaks into the model; a test set used for selection becomes a second validation set, and its score is optimistically biased (the best of many noisy evaluations, like the best of many seeds).
In this unit the test rows are not returned by any accessor: they are reachable only through `capstone_data.final_test_score(predict_fn, n_boot=1000, alpha=0.05, task=...)`, which counts its calls (`capstone_data.test_call_count(task)`), and every model-building practice asserts that the count is exactly one, after training.

**Leakage through preprocessing.** Normalization statistics, vocabulary choices, feature scalers, and k-means centroids are *fitted* quantities.
Fit them on training rows only.
Standardizing with the mean of all rows (train + validation + test) leaks held-out information into every feature.

**Seams and hashes.** This unit's practices name a **seam**, a function every training row must pass through (the model's `forward`, a feature extractor, or a `fit` call).
At that seam you can hash each incoming row with `capstone_data.row_sha256` and look it up with `capstone_data.split_of(task, row)`; any `"val"` or `"test"` result during fitting proves leakage.

**The experiment log** (`book1:metric-driven-iteration`) is a list of records written as you go: run ID, timestamp, hypothesis, configuration, seed, steps used, validation score, wall-clock time, and a one-line note ("aug std 0.03; loss curve still falling").
It records failed runs too.
It is what lets you (and a grader) reconstruct why the final configuration was chosen.

**Checkpoint 6A.** A teammate standardizes inputs with `x_all.mean(0)` computed over all 1,200 rows before splitting. Which split's information leaks, and into what?

**Checkpoint 6B.** After choosing a configuration on validation, you score it on the test set, dislike the number, tweak the learning rate, and score again. What has happened to the test set?

**Collected answers.** Validation and test rows contribute to the standardization mean (and standard deviation), so their information enters every training input and every later prediction. It has become a selection set: the second score is chosen with knowledge of the first, so it is optimistically biased and no longer an unbiased estimate; the honest fix is to report the first score (or obtain a fresh test set).

## 7. Worked laboratory: a budgeted mini-campaign with a log

The laboratory runs the Section 1 hypothesis on the inverse task, entirely on validation rows.
Hypothesis: *log magnitudes reduce the mean validation position error of the MLP by at least 10% relative to raw standardized magnitudes, at 150 Adam steps, with seeds 0 and 1.*
Every optimizer is wrapped by one `StepBudget(1000)`; the arithmetic ($2$ configurations $\times$ $2$ seeds $\times$ $150$ steps $=600$) is checked before any run.
Each run appends a record to the log, the decision rule is applied to the logged numbers, and **no test score is computed**: this is a design exercise, and the final test call belongs to the practices.

Because the preprocessing lives inside the model's `forward` (as registered buffers), the model receives exactly the rows `train_rows` returns, and the seam check hashes them.

**Checkpoint 7A.** In the laboratory, which number decides the hypothesis, and on which split is it computed?

**Checkpoint 7B.** Why are the normalization statistics stored as buffers inside the model rather than applied before calling it?

**Collected answers.** The relative change in the mean (over seeds 0 and 1) validation position error, computed on the validation rows only. So that the model's `forward` receives the clean rows exactly as returned (making the seam hash check possible), and so that the statistics move with the model and are saved in its `state_dict`.

In [ ]:
import importlib.util
from pathlib import Path

import torch
from torch import nn
import torch.nn.functional as F

SEED = 20261022
LOADER_CANDIDATES = (
    Path("../data/capstone_data.py"),
    Path("units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
    Path("book2/units/B2-024-gpu-scientific-ml-capstone/data/capstone_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("capstone_data", loader_path)
capstone_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(capstone_data)
torch.set_num_threads(1)
import time

train_x, train_y = capstone_data.train_rows("inverse")
val_x, val_y = capstone_data.val_rows("inverse")

class InverseMLP(nn.Module):
    def __init__(self, train_inputs, use_log):
        super().__init__()
        self.use_log = use_log
        feats = torch.log(train_inputs) if use_log else train_inputs
        self.register_buffer("mu", feats.mean(dim=0))          # fitted on training rows only
        self.register_buffer("sd", feats.std(dim=0))
        self.net = nn.Sequential(nn.Linear(8, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 3))
        self.seen_splits = None

    def forward(self, magnitudes):
        if self.training and self.seen_splits is None:          # the seam: hash the rows of the first
            self.seen_splits = {capstone_data.split_of("inverse", row) for row in magnitudes}  # full-batch call
        feats = torch.log(magnitudes) if self.use_log else magnitudes
        return self.net((feats - self.mu) / self.sd)

CONFIGS = {"raw": False, "log": True}
SEEDS = (0, 1)
STEPS = 150
budget = capstone_data.StepBudget(1000)
planned = len(CONFIGS) * len(SEEDS) * STEPS
assert planned <= budget.max_steps, "plan the campaign before running it"
log = []

In [ ]:
for name, use_log in CONFIGS.items():
    for seed in SEEDS:
        start = time.perf_counter()
        torch.manual_seed(seed)
        model = InverseMLP(train_x, use_log)
        optimizer = budget.wrap(torch.optim.Adam(model.parameters(), lr=3e-3))
        model.train()
        for _ in range(STEPS):
            optimizer.zero_grad(set_to_none=True)
            loss = F.mse_loss(model(train_x), train_y)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            val_err = capstone_data.position_error_per_example(model(val_x), val_y).mean().item()
        assert model.seen_splits == {"train"}, "only training rows may reach forward during fitting"
        log.append({"run": len(log), "config": name, "seed": seed, "steps": STEPS,
                    "val_error": round(val_err, 5), "seconds": round(time.perf_counter() - start, 2)})

for record in log:
    print(record)
means = {name: sum(r["val_error"] for r in log if r["config"] == name) / len(SEEDS) for name in CONFIGS}
relative = (means["raw"] - means["log"]) / means["raw"]
print("budget used:", budget.used, "of", budget.max_steps, "| means:", {k: round(v, 5) for k, v in means.items()})
print(f"relative reduction {relative:.1%} -> hypothesis", "SUPPORTED" if relative >= 0.10 else "NOT supported",
      "| test calls:", capstone_data.test_call_count("inverse"))

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *No hypothesis.* Fix: write change, metric, direction, minimum effect, and protocol before the first run.
2. *Confounded comparisons.* Fix: change one thing per ablation.
3. *Unequal budgets.* Fix: count optimizer steps with `StepBudget`; same steps per run across arms.
4. *One seed.* Fix: shared seed lists; mean and sample standard deviation; per-seed differences.
5. *Population instead of sample standard deviation.* Fix: divisor $r-1$.
6. *Selecting on the test set.* Fix: select on validation; one `final_test_score` call, after fitting.
7. *Preprocessing fitted on all rows.* Fix: fit statistics on training rows; keep them inside the model as buffers.
8. *No log, or a log of only the good runs.* Fix: record every run, including failures and restarts.

**Exam connections.** p03 chooses the baseline and single ablation for a stated hypothesis; p12 implements `run_ablation` with a mean ± sample-standard-deviation table under a fixed step budget; p22 runs a full budgeted campaign (hypothesis → configs → results → decision) with a log and one test call; p24 reads a supplied validation results table, decides which differences are real, and chooses the next experiments under a time limit.
In Round 2's open-ended problems, a short, honest design paragraph and a log are what separate a defensible score from a lucky one.

**Checkpoint 8A.** A report lists five configurations, each with one seed and different step counts. Name two pitfalls.

**Checkpoint 8B.** Which practice requires a pre-registered decision rule and a single test call?

**Collected answers.** One seed (no spread estimate) and unequal budgets (step counts differ), and possibly confounding if several things changed. p22.

## 9. Forward-only deeper topic: smarter search under a budget

Forward-only deeper topic.
When there are many knobs, a grid wastes budget on unimportant ones; **random search** samples configurations at random and, for the same budget, explores more values of the knobs that matter.
**Successive halving** starts many configurations with a small step budget, keeps the better half on validation, doubles their budget, and repeats, so most compute goes to promising runs; Hyperband runs several such brackets.
**Bayesian optimization** fits a probabilistic model of score versus configuration and chooses the next run to balance exploration and exploitation.
All of them are selection procedures, so all of them must use validation data only, and all of them make the best-found validation score optimistic; the locked test score is still taken once.
None of these methods are required, permitted shortcuts, or assessed APIs in B2-024.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Why can random search beat grid search with the same number of runs?

**Checkpoint 9B.** Does successive halving change the rule that the test set is scored once?

**Collected answers.** A grid repeats the same few values of each knob, so with one important knob most runs duplicate its values; random sampling tries a new value of every knob in every run. No: it is a validation-only selection procedure; the final model is scored on the test set once.